<a href="https://colab.research.google.com/github/sandeep0608/spatial-Recon/blob/main/stage12_mast3r_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1-2 — MASt3R reconstruction (uncalibrated photos → point cloud)

Part of the **spatial-recon** pipeline. This notebook runs the GPU-heavy stages in the cloud
and downloads `aligned_pointcloud.ply`, which you then feed to the local `scripts/stage34_mesh.py`.

**Before you start:** set **Runtime → Change runtime type → GPU** (a free T4 is fine).

> MASt3R is licensed CC BY-NC-SA 4.0 (non-commercial). For commercial use, switch to VGGT (see the README).


### 1. Check the GPU

In [ ]:
!nvidia-smi

Fri Nov  6 15:36:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Install MASt3R + Open3D
Clones MASt3R (with its DUSt3R submodule) and installs dependencies. Takes a couple of minutes.
Model weights download automatically from Hugging Face the first time the model is loaded.

In [ ]:
!git clone --recursive https://github.com/naver/mast3r
%cd mast3r
!pip install -q -r requirements.txt
!pip install -q open3d
print("install done")

Cloning into 'mast3r'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 269 (delta 115), reused 109 (delta 109), pack-reused 99 (from 1)
Receiving objects: 100% (269/269), 3.59 MiB | 43.78 MiB/s, done.
Resolving deltas: 100% (151/151), done.
Submodule 'dust3r' (https://github.com/naver/dust3r) registered for path 'dust3r'
Cloning into '/content/mast3r/dust3r'...
remote: Enumerating objects: 611, done.        
remote: Total 611 (delta 0), reused 0 (delta 0), pack-reused 611 (from 1)        
Receiving objects: 100% (611/611), 756.60 KiB | 16.45 MiB/s, done.
Resolving deltas: 100% (355/355), done.
Submodule path 'dust3r': checked out '3cc8c88c413bb9e34c41db0e0eef99c2ee010b12'
Submodule 'croco' (https://github.com/naver/croco) registered for path 'dust3r/croco'
Cloning into '/content/mast3r/dust3r/croco'...
remote: Enumerating objects: 198, done.        
remote: Counting objects: 100% (8

### 3. Upload your photos
Run this, then pick 8-30 images of ONE place. They go into `input_images/`.
(Tip: overlapping viewpoints + similar lighting + few people/cars = much better results.)

In [ ]:
import os
from google.colab import files

os.makedirs('input_images', exist_ok=True)
uploaded = files.upload()
for fn in list(uploaded.keys()):
    os.replace(fn, os.path.join('input_images', fn))
print("uploaded:", sorted(os.listdir('input_images')))

Saving caption (12).jpg to caption (12).jpg
Saving i-had-an-incredible-time.jpg to i-had-an-incredible-time.jpg
Saving caption (11).jpg to caption (11).jpg
Saving caption (10).jpg to caption (10).jpg
Saving caption (9).jpg to caption (9).jpg
Saving caption (8).jpg to caption (8).jpg
Saving caption (7).jpg to caption (7).jpg
Saving caption (6).jpg to caption (6).jpg
Saving caption (5).jpg to caption (5).jpg
Saving caption (4).jpg to caption (4).jpg
Saving bhutanatha-temple.jpg to bhutanatha-temple.jpg
Saving wonderful-sculptures.jpg to wonderful-sculptures.jpg
Saving caption (3).jpg to caption (3).jpg
Saving caption (2).jpg to caption (2).jpg
Saving caption (1).jpg to caption (1).jpg
Saving caption.jpg to caption.jpg
Saving badami-cave-temples.jpg to badami-cave-temples.jpg
uploaded: ['badami-cave-temples.jpg', 'bhutanatha-temple.jpg', 'caption (1).jpg', 'caption (10).jpg', 'caption (11).jpg', 'caption (12).jpg', 'caption (2).jpg', 'caption (3).jpg', 'caption (4).jpg', 'caption (5).jpg'

In [ ]:
# Install roma and the required dust3r/mast3r sub-packages
!pip install -q roma einops trimesh open3d scipy matplotlib tqdm opencv-python huggingface_hub "pyglet<2"

# Ensure requirements files from both repos are installed
!pip install -q -r /content/mast3r/requirements.txt
!pip install -q -r /content/mast3r/dust3r/requirements.txt

In [ ]:
import roma, open3d
print("All dependencies installed successfully!")

All dependencies installed successfully!


In [ ]:
import os; print(os.getcwd())
!ls /content/mast3r 2>/dev/null && echo "repo still here" || echo "repo gone - full reset"

/content/mast3r
assets		    docker		       make_pairs.py	 train.py
CHECKPOINTS_NOTICE  dust3r		       mast3r		 visloc.py
demo_dust3r_ga.py   input_images	       NOTICE
demo_glomap.py	    kapture_mast3r_mapping.py  README.md
demo.py		    LICENSE		       requirements.txt
repo still here


### 4. Load the MASt3R model

In [ ]:
import torch
from mast3r.model import AsymmetricMASt3R
import mast3r.utils.path_to_dust3r  # noqa: makes the dust3r submodule importable
from dust3r.inference import inference
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode
from dust3r.utils.device import to_numpy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AsymmetricMASt3R.from_pretrained(
    "naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric").to(device)
print("model loaded on", device)

Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/content/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.75GB            

model.safetensors: downloading bytes:           |  0.00B            

model loaded on cuda


### 5. Run inference + global alignment
This is Stage 2: pairwise 3D prediction, then `PointCloudOptimizer` locks all pairs into one
metric coordinate space.

- `SCENE_GRAPH='complete'` compares every pair (best quality). It scales **quadratically**, so for
  more than ~15 images switch to `'swin-3'` (sliding window) to avoid running out of GPU memory.
- `CONF_THR` is the confidence cutoff for keeping points (higher = stricter, less sky/junk).

In [ ]:
import glob

SCENE_GRAPH = 'swin-3'   # many images (>15)? use 'swin-3'
NITER       = 300
LR          = 0.01
SCHEDULE    = 'cosine'
CONF_THR    = 3.0

image_paths = sorted(glob.glob('input_images/*'))
assert len(image_paths) >= 2, "Need at least 2 images in input_images/"
print(f"{len(image_paths)} images")

images = load_images(image_paths, size=512)
pairs  = make_pairs(images, scene_graph=SCENE_GRAPH, prefilter=None, symmetrize=True)
print(f"{len(pairs)} pairs")

output = inference(pairs, model, device, batch_size=1, verbose=True)

scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
loss  = scene.compute_global_alignment(init="mst", niter=NITER, schedule=SCHEDULE, lr=LR)
print("global alignment loss:", float(loss))

17 images
>> Loading a list of 17 images
 - adding input_images/badami-cave-temples.jpg with resolution 2000x1125 --> 512x288
 - adding input_images/bhutanatha-temple.jpg with resolution 1400x788 --> 512x288
 - adding input_images/caption (1).jpg with resolution 850x1133 --> 384x512
 - adding input_images/caption (10).jpg with resolution 1400x654 --> 512x224
 - adding input_images/caption (11).jpg with resolution 1050x1311 --> 400x512
 - adding input_images/caption (12).jpg with resolution 1024x768 --> 512x384
 - adding input_images/caption (2).jpg with resolution 850x1133 --> 384x512
 - adding input_images/caption (3).jpg with resolution 1400x654 --> 512x224
 - adding input_images/caption (4).jpg with resolution 1133x850 --> 512x384
 - adding input_images/caption (5).jpg with resolution 1100x1956 --> 288x512
 - adding input_images/caption (6).jpg with resolution 1100x2355 --> 224x512
 - adding input_images/caption (7).jpg with resolution 1100x2355 --> 224x512
 - adding input_images/ca

  0%|          | 0/102 [00:00<?, ?it/s]/content/mast3r/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/content/mast3r/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/content/mast3r/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 102/102 [01:27<00:00,  1.17it/s]


 init edge (11*,10*) score=np.float64(10.108955383300781)
 init edge (12*,11) score=np.float64(1.8146573305130005)
 init edge (12,15*) score=np.float64(5.927613735198975)
 init edge (15,14*) score=np.float64(2.8615126609802246)
 init edge (15,1*) score=np.float64(2.553415298461914)
 init edge (13*,14) score=np.float64(2.1496880054473877)
 init edge (15,16*) score=np.float64(1.6825520992279053)
 init edge (14,0*) score=np.float64(3.7782602310180664)
 init edge (2*,0) score=np.float64(3.782468795776367)
 init edge (5*,2) score=np.float64(3.5784034729003906)
 init edge (5,8*) score=np.float64(3.2090890407562256)
 init edge (8,7*) score=np.float64(2.7430667877197266)
 init edge (5,3*) score=np.float64(2.2452852725982666)
 init edge (3,4*) score=np.float64(7.823696136474609)
 init edge (8,6*) score=np.float64(5.3395161628723145)
 init edge (9*,6) score=np.float64(2.0611398220062256)
 init loss = 0.14569281041622162
Global alignement - optimizing for:
['pw_poses', 'im_depthmaps', 'im_poses',

  0%|          | 0/300 [00:00<?, ?it/s]/content/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:366: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), lr
100%|██████████| 300/300 [01:12<00:00,  4.12it/s, lr=1.27413e-06 loss=0.0231727]

global alignment loss: 0.023172687739133835


### 6. Build & export the colored point cloud
Keeps only confident pixels (drops most sky/background), then writes and downloads
`aligned_pointcloud.ply`.

In [ ]:
import numpy as np
import open3d as o3d

scene.min_conf_thr = float(CONF_THR)
pts3d = to_numpy(scene.get_pts3d())   # list of HxWx3
masks = to_numpy(scene.get_masks())   # list of HxW bool (confidence > threshold)
imgs  = scene.imgs                    # list of HxWx3 in [0,1]

pts = np.concatenate([p[m].reshape(-1, 3) for p, m in zip(pts3d, masks)], axis=0)
col = np.concatenate([im[m].reshape(-1, 3) for im, m in zip(imgs, masks)], axis=0)
print("kept", len(pts), "points")

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
pcd.colors = o3d.utility.Vector3dVector(np.clip(col, 0, 1).astype(np.float64))
o3d.io.write_point_cloud("aligned_pointcloud.ply", pcd)
print("saved aligned_pointcloud.ply")

from google.colab import files
files.download("aligned_pointcloud.ply")

kept 1005443 points
saved aligned_pointcloud.ply


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Next step — local meshing
Put the downloaded `aligned_pointcloud.ply` in your project folder and run:

```bash
python scripts/stage34_mesh.py --input aligned_pointcloud.ply
```

That produces `output_mesh/reconstructed_game_mesh.obj` for Blender → your game engine.
If the cloud looks too sparse or too noisy, re-run cell 5 with a different `CONF_THR`
(2.0 = keep more, 5.0 = keep only the most confident).